# chemagent — Debugging Notebook

End-to-end pipeline using the consolidated `chemagent_mcp.py` server (16 tools).

**Preferred workflow (data stays on disk)**
```
find_datasets → load_dataset → compute_features → split_dataset
  → train_model (non-blocking) → check_training (poll)
  → plot_classification_results
```
**Shortcut (load/featurize/split synchronously, then trains in background)**
```
job = run_pipeline("data/datasets/...", algorithm="RFC", task="classification")
result = check_training(job["job_id"])   # poll every 15-30 s
```


## 1. Environment setup

In [1]:
import sys
from pathlib import Path

# Resolve workspace root (works whether cwd is notebooks/ or the repo root)
_ws_root = Path.cwd()
if not (_ws_root / "src").exists():
    _ws_root = _ws_root.parent

_servers_dir = _ws_root / "src" / "chemagent" / "servers"

for _p in [str(_ws_root), str(_servers_dir)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

_save_dir = _ws_root / "notebooks" / "debug_outputs"
_save_dir.mkdir(exist_ok=True)

print(f"Workspace root : {_ws_root}")
print(f"Servers dir    : {_servers_dir}")
print(f"Save dir       : {_save_dir}")

Workspace root : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability
Servers dir    : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\src\chemagent\servers
Save dir       : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\notebooks\debug_outputs


## 2. Imports

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

from src.chemagent.servers.chemagent_mcp import (
    # ── Dataset tools (MCP) ────────────────────────────────────
    find_datasets,
    list_loaded_datasets,
    list_featurizers,
    load_dataset,
    dataset_status,
    compute_features,
    split_dataset,
    # ── ML tools (MCP) ─────────────────────────────────────────
    get_ml_info,
    train_model,
    check_training,
    export_predictions,
    # ── Internal helpers (not MCP tools, useful for notebook) ──
    evaluate_classification,
)
from src.chemagent.plots import (
    plot_confusion_matrix,
    plot_roc_curve,
    plot_pr_curve,
    plot_metric_bar,
    plot_feature_importance,
    plot_class_distribution,
    plot_split_statistics,
)
import joblib, numpy as np, time


## 3. Discover available options

In [3]:
find_datasets()


{'datasets': ['chembl_activity_data_O00329_P42336.csv',
  'chembl_activity_data_O00329_P48736.csv',
  'chembl_activity_data_P42336_P48736.csv'],
 'count': 3,
 'directory': 'C:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data\\datasets'}

In [4]:
load_dataset('/data/datasets/chembl_activity_data_O00329_P48736.csv')

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'n_samples': 1277,
 'columns': ['smiles', 'class_label', 'pPot_diff', 'target_pair', 'cid'],
 'label_col': 'class_label',
 'label_stats': {'mean': 0.28191072826938135,
  'std': 0.5268901818796558,
  'min': 0.0,
  'max': 2.0,
  'unique_values': 3},
 'has_smiles': True,
 'has_precomputed_features': False,
 'loaded': True,
 'next_step': "Call featurize_dataset(dataset_id='chembl_activity_data_O00329_P48736', method='ECFP', radius=2, n_bits=2048) to compute fingerprints server-side, then split_prepared_dataset().",
 'smiles_sample': ['CC(NC(=O)c1c(N)nn2cccnc12)c1cc2cccc(Cl)c2c(=O)n1-c1ccc(O)cc1',
  'Cc1cccc(NS(=O)(=O)c2ccc(C)c(-c3cnc(N)c(-c4cnn(C)c4)c3)c2)n1',
  'Cc1ccc(S(=O)(=O)NCC(C)(C)O)cc1-c1cnc(N)c(-c2ncco2)c1']}

In [5]:
list_featurizers()

{'ECFP': {'parameters': {'n_bits': '2048', 'radius': '2', 'sparse': 'False'},
  'description': 'Generate ECFP (Morgan) bit-vector fingerprints from SMILES strings.'},
 'MACCS': {'parameters': {},
  'description': 'Generate 166-bit MACCS structural-key fingerprints from SMILES strings.'},
 'RDKitFP': {'parameters': {'n_bits': '2048',
   'min_path': '1',
   'max_path': '7'},
  'description': 'Generate RDKit topological (path-based) fingerprints.'},
 'AtomPairFP': {'parameters': {'n_bits': '2048'},
  'description': 'Generate atom-pair fingerprints.'},
 'TopologicalTorsionFP': {'parameters': {'n_bits': '2048'},
  'description': 'Generate topological-torsion fingerprints.'}}

In [6]:
# Returns algorithms + hyperparameter grids + recommended metrics in one call
get_ml_info()

{'algorithms': {'RFR': {'name': 'Random Forest Regressor',
   'task_type': 'regression',
   'hyperparameters': {'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]},
   'supports_multiclass': False,
   'description': 'Ensemble of decision trees for regression tasks'},
  'RFC': {'name': 'Random Forest Classifier',
   'task_type': 'classification',
   'hyperparameters': {'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]},
   'supports_multiclass': True,
   'description': 'Ensemble of decision trees for classification, handles multi-class'},
  'SVC': {'name': 'Support Vector Classifier',
   'task_type': 'classification',
   'hyperparameters': {'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']},
   'supports_multiclass': True,
   'description': 'SVM classifier with RBF/linear kern

## 4. Load dataset

In [7]:
dataset_info = load_dataset("data/datasets/chembl_activity_data_O00329_P48736.csv")
dataset_info

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'n_samples': 1277,
 'columns': ['smiles', 'class_label', 'pPot_diff', 'target_pair', 'cid'],
 'label_col': 'class_label',
 'label_stats': {'mean': 0.28191072826938135,
  'std': 0.5268901818796558,
  'min': 0.0,
  'max': 2.0,
  'unique_values': 3},
 'has_smiles': True,
 'has_precomputed_features': False,
 'loaded': True,
 'next_step': "Call featurize_dataset(dataset_id='chembl_activity_data_O00329_P48736', method='ECFP', radius=2, n_bits=2048) to compute fingerprints server-side, then split_prepared_dataset().",
 'smiles_sample': ['CC(NC(=O)c1c(N)nn2cccnc12)c1cc2cccc(Cl)c2c(=O)n1-c1ccc(O)cc1',
  'Cc1cccc(NS(=O)(=O)c2ccc(C)c(-c3cnc(N)c(-c4cnn(C)c4)c3)c2)n1',
  'Cc1ccc(S(=O)(=O)NCC(C)(C)O)cc1-c1cnc(N)c(-c2ncco2)c1']}

In [56]:
dataset_status(dataset_info["dataset_id"])

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'loaded': True,
 'raw_data': {'n_samples': 1277,
  'columns': ['smiles', 'class_label', 'pPot_diff', 'target_pair', 'cid'],
  'label_col': 'class_label',
  'smiles_col': 'smiles',
  'id_col': None},
 'prepared': True,
 'ml_ready': {'n_samples': 1277,
  'n_features': 2048,
  'label_column': 'class_label',
  'has_smiles': True,
  'has_cid': False}}

## 5. Featurize (server-side — no large arrays transferred)

In [57]:
featurized = compute_features(
    dataset_info["dataset_id"],
    method="ECFP",
    radius=2,
    n_bits=2048,
)
featurized

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'method': 'ECFP',
 'n_samples': 1277,
 'n_features': 2048,
 'label_stats': {'mean': 0.28191072826938135,
  'std': 0.5268901818796558,
  'min': 0.0,
  'max': 2.0,
  'unique_values': 3},
 'prepared': True,
 'next_step': "Call split_dataset('chembl_activity_data_O00329_P48736', train_size=0.7, val_size=0.0, test_size=0.3, stratified=True) to create splits."}

## 6. Split

In [58]:
data_splits = split_dataset(
    dataset_info["dataset_id"],
    train_size=0.6,
    val_size=0.0,
    test_size=0.4,
    stratified=True,
    split_type="random",
    save_path=str(_save_dir / "data_splits.csv")
)
data_splits

{'train': {'n_samples': 766,
  'indices': [699,
   1087,
   939,
   987,
   331,
   314,
   66,
   105,
   401,
   352,
   213,
   159,
   387,
   96,
   316,
   240,
   983,
   457,
   334,
   1177,
   206,
   1179,
   426,
   465,
   200,
   693,
   1257,
   1133,
   588,
   374,
   1175,
   202,
   230,
   1115,
   207,
   435,
   794,
   417,
   131,
   87,
   244,
   152,
   900,
   1230,
   390,
   1171,
   567,
   1240,
   565,
   226,
   869,
   1016,
   81,
   890,
   889,
   255,
   781,
   1094,
   582,
   347,
   99,
   15,
   419,
   755,
   471,
   11,
   573,
   450,
   430,
   497,
   908,
   1037,
   991,
   653,
   1106,
   409,
   50,
   367,
   702,
   220,
   1169,
   1131,
   504,
   775,
   1190,
   18,
   1117,
   392,
   632,
   1158,
   1275,
   1202,
   1039,
   1045,
   227,
   406,
   411,
   641,
   93,
   828,
   74,
   345,
   436,
   327,
   1046,
   640,
   1201,
   349,
   1211,
   929,
   1208,
   985,
   1261,
   866,
   773,
   286,
   518,
   712,

## 6b. Dataset & split visualisation

In [ ]:
# MCP equivalent (one call):
#   plot_dataset_info(dataset_info["dataset_id"], split_file_path=data_splits["saved_to"])

_split = joblib.load(data_splits["saved_to"])
_all_labels = np.concatenate([_split["train_labels"], _split["test_labels"]])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_class_distribution(
    _all_labels,
    title="Class Distribution (full dataset)",
    ax=axes[0],
)
plot_split_statistics(
    data_splits["statistics"],
    title="Train / Test Split",
    ax=axes[1],
)
fig.tight_layout()
plt.show()

## 7. Train model (tune → train → evaluate)

**MCP pattern** (non-blocking, used by the LLM agent):
```python
job = train_model(split_file_path=..., algorithm="RFC", task="classification")
result = check_training(job["job_id"])   # poll until status != "running"
```

**Notebook cell below** uses the internal helper `build_model_from_split_file` (blocking, convenient for direct exploration).

In [59]:
# Non-blocking MCP approach (mirrors what the LLM agent does)
job = train_model(
    split_file_path=data_splits["saved_to"],
    algorithm="RFC",
    task="classification",
    opt_metric="balanced_accuracy",
    model_save_path=str(_save_dir / "trained_model.pkl")
)
print("Job submitted:", job["job_id"])

while True:
    status = check_training(job["job_id"])
    if status["status"] != "running":
        break
    print(f"Training... {status['elapsed_seconds']:.1f}s elapsed")
    time.sleep(5)

model_result = status["result"]
print("Status:", status["status"])
model_result

Job submitted: 9f2cd677-bf97-4e1b-8c00-78968125b195
Training... 0.0s elapsed
Training... 5.0s elapsed
Training... 10.0s elapsed
Status: completed


{'algorithm': 'RFC',
 'task': 'classification',
 'cv_fold': 5,
 'opt_metric': 'balanced_accuracy',
 'best_params': {'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 200},
 'cv_best_score': 0.8045943426136393,
 'model_path': 'c:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\notebooks\\debug_outputs\\trained_model.pkl',
 'hyperparameters_searched': {'n_estimators': [50, 100, 200],
  'max_features': ['sqrt', 'log2'],
  'min_samples_split': [2, 5, 10],
  'min_samples_leaf': [1, 2, 4]},
 'train_evaluation': {'target': 'train',
  'algorithm': 'RFC',
  'overall_metrics': {'MCC': 1.0,
   'Accuracy': 1.0,
   'BA': 1.0,
   'F1 macro': 1.0,
   'F1 weighted': 1.0},
  'per_class_metrics': {'Class_0': {'Precision': 1.0,
    'Recall': 1.0,
    'F1': 1.0,
    'Support': 579},
   'Class_1': {'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0, 'Support': 158},
   'Class_2': {'Precision': 1.0, 'Recall': 1

## 8. Inspect results

In [ ]:
print("Best params  :", model_result["best_params"])
print("CV best score:", model_result["cv_best_score"])
print("Model saved  :", model_result["model_path"])

In [ ]:
model_result["train_evaluation"]

In [ ]:
model_result["test_evaluation"]

## 9. Standalone evaluation (optional)

Uses internal helpers `evaluate_classification` / `evaluate_regression` (not MCP tools — available for direct notebook use).

In [ ]:
split    = joblib.load(data_splits["saved_to"])
model    = joblib.load(model_result["model_path"])
X_test   = np.array(split["test_features"])
y_test   = split["test_labels"].tolist()

y_pred   = model.predict(X_test).tolist()
y_proba  = model.predict_proba(X_test).tolist()

# internal helper — still available for custom evaluation in the notebook
evaluate_classification(
    labels=y_test,
    predictions=y_pred,
    probabilities=y_proba,
)

## 9b. Classification plots

**MCP equivalent** (loads model + split from disk, generates all figures in one call):
```python
plot_classification_results(model_result["model_path"], data_splits["saved_to"])
```

Cells below call the underlying plot functions directly for finer notebook control.

In [ ]:
_n_classes  = len(set(y_test))
_is_binary  = _n_classes == 2

# binary eval → flat dict; multiclass eval → nested, scalars in "overall_metrics"
_test_metrics = model_result["test_evaluation"]
_scalar_metrics = _test_metrics.get("overall_metrics", _test_metrics)

if _is_binary:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    plot_confusion_matrix(
        y_test, y_pred,
        title="Confusion Matrix (test)", ax=axes[0],
    )
    plot_roc_curve(
        y_test, [p[1] for p in y_proba],
        title="ROC Curve (test)", ax=axes[1],
    )
    plot_pr_curve(
        y_test, [p[1] for p in y_proba],
        title="Precision-Recall Curve (test)", ax=axes[2],
    )
else:
    # Multiclass: confusion matrix + overall metric bar
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_confusion_matrix(
        y_test, y_pred,
        title=f"Confusion Matrix — {_n_classes} classes (test)", ax=axes[0],
    )
    plot_metric_bar(_scalar_metrics, title="Test Set Metrics (overall)", ax=axes[1])

fig.tight_layout()
plt.show()

In [ ]:
# Evaluation metrics bar chart
# binary eval → flat dict; multiclass eval → nested, scalars in "overall_metrics"
_eval = model_result["test_evaluation"]
_scalars = _eval.get("overall_metrics", _eval)
plot_metric_bar(_scalars, title="Test Set Metrics")
#plt.show()

In [ ]:
# Top-20 feature importances (RFC / tree-based models only)
_model = joblib.load(model_result["model_path"])
plot_feature_importance(_model, top_n=20, title="Top 20 Feature Importances")
#plt.show()

## 9c. Export predictions to CSV (MCP tool)

`export_predictions` writes a per-sample CSV with (where available):

| Column | Description |
|---|---|
| `cid` | Compound ID (from the original dataset, if present) |
| `smiles` | SMILES string (if present) |
| `true_label` | Ground-truth label |
| `predicted_label` | Model prediction |
| `prob_class_0`, `prob_class_1` | Class probabilities (classification only) |

For regression tasks `predicted_label` is replaced by `predicted_value`. A `.pkl` metrics file is saved alongside the CSV. Both land in `<session>/results/`.

In [ ]:
export_result = export_predictions(
    model_path=model_result["model_path"],
    split_file_path=data_splits["saved_to"],
    task="classification",
    split="test",          # default
)
export_result

In [ ]:
import pandas as pd
pd.read_csv(export_result["csv_path"]).head(10)


## 10. Scratch / exploratory

In [ ]:
# Scratch cell — free-form exploration
